<a href="https://colab.research.google.com/github/ppphx/Time-Series/blob/main/practice3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from scipy.stats import boxcox
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('passengers.csv')

# Преобразование столбца 'Month' в datetime и установка его в качестве индекса
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)
df.sort_index(inplace=True)

print(df.head())

2. Разбиение

In [ ]:
# Определяем размеры выборок
n = len(df)
train_size = int(n * 0.75)
validate_size = int(n * 0.15)

# Создаем выборки
train_df = df[:train_size]
validate_df = df[train_size:train_size + validate_size]
test_df = df[train_size + validate_size:]

print(f"Размер обучающей выборки: {len(train_df)}")
print(f"Размер валидационной выборки: {len(validate_df)}")
print(f"Размер тестовой выборки: {len(test_df)}")

3. Подготовка данных для AR

In [ ]:
# Применяем преобразование Бокса-Кокса ко всему ряду, чтобы найти лямбду, используем весь ряд, чтобы найти оптимальную лямбду.
transformed_series, best_lambda = boxcox(df['Passengers'])

# Функция для преобразования Бокса-Кокса с заданной лямбдой
def boxcox_transform(series, lam):
    if lam == 0:
        return np.log(series)
    else:
        return (series**lam - 1) / lam

# Преобразуем каждую выборку
train_transformed = boxcox_transform(train_df['Passengers'], best_lambda)
validate_transformed = boxcox_transform(validate_df['Passengers'], best_lambda)
test_transformed = boxcox_transform(test_df['Passengers'], best_lambda)

# Удаляем тренд (дифференцирование)
train_stationary = np.diff(train_transformed)
# Для прогноза нам понадобится последнее значение обучающей выборки в преобразованном виде
last_train_transformed = train_transformed.iloc[-1]

# Валидационная и тестовая выборки также нужно сделать стационарными относительно предыдущих значений
# Для валидационной выборки: вычитаем предыдущее значение из обучающей
validate_stationary = validate_transformed - train_transformed.iloc[-1]

# Для тестовой выборки: вычитаем предыдущее значение.
test_stationary = test_transformed - validate_transformed.iloc[-1]

# Отобразим стационарный ряд обучающей выборки
plt.figure(figsize=(10, 4))
plt.plot(train_stationary)
plt.title('Стационарный временной ряд (Обучающая выборка после преобразований)')
plt.show()

4. Обучение AR

In [ ]:
from statsmodels.tools.sm_exceptions import HessianInversionWarning
warnings.filterwarnings("ignore", category=HessianInversionWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Оценка моделей AR с разными p
best_aic = np.inf
best_p = 0
aic_values = {}

for p in range(1, 15):  # Проверим p от 1 до 14
    try:
        model = AutoReg(train_stationary, lags=p, old_names=False).fit()
        aic = model.aic
        aic_values[p] = aic
        if aic < best_aic:
            best_aic = aic
            best_p = p
    except:
        continue

print(f"Лучший порядок модели: p = {best_p}, AIC = {best_aic}")

# Обучаем лучшую модель
final_model = AutoReg(train_stationary, lags=best_p, old_names=False).fit()
print(final_model.summary())

5. Прогноз

In [ ]:
# Функция для обратного преобразования Бокса-Кокса
def inverse_boxcox(transformed_series, lam, original_series):
    if lam == 0:
        return np.exp(transformed_series)
    else:
        return (transformed_series * lam + 1) ** (1/lam)

# Делаем прогноз на тестовую выборку
forecast_steps = len(test_df)
# Берем последнее
last_observed = train_transformed.iloc[-1]

# Прогнозируем приращения
forecast_diff = final_model.forecast(steps=forecast_steps)

# Восстанавливаем преобразованный ряд, начиная с последнего наблюдения
forecast_transformed = []
current = last_observed
for diff in forecast_diff:
    current = current + diff
    forecast_transformed.append(current)
forecast_transformed = np.array(forecast_transformed)

# Преобразуем прогноз обратно в исходный масштаб
forecast_original = inverse_boxcox(forecast_transformed, best_lambda, None)

# Преобразуем фактические значения тестовой выборки в исходный масштаб
actual_test = test_df['Passengers'].values

print("Прогноз на тестовую выборку (первые 5 значений):")
print(forecast_original[:5])
print("\nФактические значения тестовой выборки (первые 5):")
print(actual_test[:5])

6. Визуализация

In [ ]:
# Создаем массив индексов для графика
train_dates = train_df.index
validate_dates = validate_df.index
test_dates = test_df.index
forecast_dates = test_dates  # Прогноз на тот же период, что и тестовая выборка

# Визуализация
plt.figure(figsize=(12, 6))
plt.plot(train_dates, train_df['Passengers'], label='Обучающая выборка', color='blue')
plt.plot(validate_dates, validate_df['Passengers'], label='Валидационная выборка', color='orange')
plt.plot(test_dates, actual_test, label='Фактические данные (тест)', color='green')
plt.plot(forecast_dates, forecast_original, label='Прогноз AR модели', color='red', linestyle='--')
plt.title('Прогнозирование временного ряда пассажиров с помощью AR модели')
plt.xlabel('Дата')
plt.ylabel('Количество пассажиров')
plt.legend()
plt.grid(True)
plt.show()

# Расчет метрик ошибки
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(actual_test, forecast_original)
rmse = np.sqrt(mean_squared_error(actual_test, forecast_original))
mape = np.mean(np.abs((actual_test - forecast_original) / actual_test)) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

**Вывод:**


*   Была построена авторегрессионная модель (AR) для прогнозирования временного ряда AirPassengers. Для обеспечения стационарности данных применялись несколько преобразований: сначала использовалось преобразование Бокса-Кокса с целью стабилизации дисперсии, а затем выполнялось дифференцирование для устранения трендовых составляющих.

*   Оптимальный порядок модели (p) определялся с помощью минимизации информационного критерия AIC. В рассматриваемом наборе данных наилучшим оказался порядок p=13.

*   Полученная модель демонстрирует удовлетворительные результаты на тестовой выборке: MAPE составляет примерно 3.3%, что указывает на достаточно высокую точность прогнозирования, отражающую среднюю относительную ошибку около 3%. Визуальный анализ графика прогноза показывает, что он в целом следует за фактическими значениями, хотя наблюдаются незначительные расхождения в области пиков.

*   Несмотря на свою относительную простоту, модель AR эффективно справляется с прогнозированием данного временного ряда, характеризующегося выраженной сезонностью и трендом. Применение последовательных преобразований — преобразования Бокса-Кокса и дифференцирования — оказалось важным условием для успешного построения модели.



